
# Fairness Metrics Mini-Lab

**Day 4 — AI Security & Legal Compliance · Practical 4 of 4 · Companion to the "AI Ethics —
Bias, Fairness & Transparency" deck**

> **Running in Google Colab:** works on the default **CPU runtime** — pure numpy/pandas, no
> GPU, no API calls.

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Compute Demographic Parity and Equalized Odds by hand on a toy dataset
2. See, with your own numbers, that the SAME system can score "fair" by one metric and
   "unfair" by another
3. Understand why choosing a fairness definition is a judgment call, not a default setting

## Why This Matters for a Law Firm

This is the deck's mini-lab example made concrete and runnable — a toy AI system screening
applications, evaluated two different ways, with genuinely conflicting results. This is exactly
the kind of analysis a firm advising a client on a screening tool's fairness would need to run
and be able to explain.

## Notebook Workflow

```mermaid
flowchart TD
    A["Toy screening dataset\n(2 groups, ground truth + predictions)"] --> B["Demographic Parity\ncalculation"]
    A --> C["Equalized Odds\ncalculation"]
    B --> D["Compare results:\ndo they agree?"]
    C --> D



## Section 1 — Setup and the Toy Dataset

We build a toy dataset mirroring the deck's mini-lab: two applicant groups, with ground-truth
qualification and the AI system's actual approve/reject predictions. The underlying
QUALIFICATION rates genuinely differ between the two groups in this toy example -- deliberately,
so we can see how different fairness metrics handle that difference differently.


In [ ]:

%pip install -q pandas numpy

import pandas as pd
import numpy as np

np.random.seed(42)

def generate_toy_data(n, qualified_rate, approval_bias=0.0):
    # ground_truth: 1 = genuinely qualified, 0 = not qualified
    # prediction: 1 = system approved, 0 = system rejected
    ground_truth = np.random.binomial(1, qualified_rate, n)
    # The system's prediction correlates with ground truth (it's not random), with some noise,
    # plus an optional bias term to simulate the system being systematically stricter/looser
    # for this group.
    approval_prob = np.clip(ground_truth * 0.8 + 0.1 + approval_bias, 0, 1)
    prediction = np.random.binomial(1, approval_prob)
    return ground_truth, prediction

# Group A: 100 applicants, higher underlying qualification rate (40%)
group_a_truth, group_a_pred = generate_toy_data(n=100, qualified_rate=0.40)

# Group B: 100 applicants, lower underlying qualification rate (20%) -- a real difference
# in the toy data's ground truth, standing in for a genuine historical disparity
group_b_truth, group_b_pred = generate_toy_data(n=100, qualified_rate=0.20)

df = pd.DataFrame({
    "group": ["A"] * 100 + ["B"] * 100,
    "ground_truth": np.concatenate([group_a_truth, group_b_truth]),
    "prediction": np.concatenate([group_a_pred, group_b_pred]),
})

print(df.groupby("group")[["ground_truth", "prediction"]].mean())
print(f"\nTotal applicants: {len(df)}")



## Section 2 — Demographic Parity

Demographic Parity asks: is the APPROVAL RATE equal across groups, regardless of the
underlying qualification rate?

`Demographic Parity Difference = P(prediction=1 | group=A) - P(prediction=1 | group=B)`

A value near 0 means the two groups are approved at similar rates.


In [ ]:

def demographic_parity(df, group_col="group", pred_col="prediction"):
    rates = df.groupby(group_col)[pred_col].mean()
    return rates

approval_rates = demographic_parity(df)
print("Approval rate by group:")
print(approval_rates)

dp_difference = approval_rates["A"] - approval_rates["B"]
print(f"\nDemographic Parity Difference (A - B): {dp_difference:.3f}")
print("(Closer to 0 = more 'fair' by this specific definition)")



## Section 3 — Equalized Odds

Equalized Odds asks a stricter question: among applicants who ARE actually qualified, is the
TRUE POSITIVE RATE (correctly approved) equal across groups? And among applicants who are NOT
qualified, is the FALSE POSITIVE RATE (incorrectly approved) equal across groups?

This is a fundamentally different question than Demographic Parity -- it conditions on the
ACTUAL ground truth, not just the raw approval rate.


In [ ]:

def equalized_odds(df, group_col="group", truth_col="ground_truth", pred_col="prediction"):
    results = {}
    for group in df[group_col].unique():
        group_df = df[df[group_col] == group]

        qualified = group_df[group_df[truth_col] == 1]
        true_positive_rate = qualified[pred_col].mean() if len(qualified) > 0 else float("nan")

        not_qualified = group_df[group_df[truth_col] == 0]
        false_positive_rate = not_qualified[pred_col].mean() if len(not_qualified) > 0 else float("nan")

        results[group] = {"TPR": true_positive_rate, "FPR": false_positive_rate}
    return results

eo_results = equalized_odds(df)
for group, rates in eo_results.items():
    print(f"Group {group}: TPR={rates['TPR']:.3f}, FPR={rates['FPR']:.3f}")

tpr_diff = eo_results["A"]["TPR"] - eo_results["B"]["TPR"]
fpr_diff = eo_results["A"]["FPR"] - eo_results["B"]["FPR"]
print(f"\nTPR difference (A - B): {tpr_diff:.3f}")
print(f"FPR difference (A - B): {fpr_diff:.3f}")
print("(Both closer to 0 = more 'fair' by THIS definition)")



## Section 4 — The Core Lesson: Do the Two Metrics Agree?

Compare the Demographic Parity difference against the Equalized Odds differences directly.


In [ ]:

print("SUMMARY\n" + "-" * 50)
print(f"Demographic Parity difference:  {dp_difference:+.3f}")
print(f"Equalized Odds TPR difference:  {tpr_diff:+.3f}")
print(f"Equalized Odds FPR difference:  {fpr_diff:+.3f}")

print("\nInterpretation:")
if abs(dp_difference) < 0.05 and (abs(tpr_diff) > 0.1 or abs(fpr_diff) > 0.1):
    print("  Demographic Parity looks roughly satisfied (approval RATES are close),")
    print("  but Equalized Odds is clearly violated (error RATES differ substantially")
    print("  between groups, conditioned on actual qualification).")
    print("  -> This system would be called 'fair' under one definition and")
    print("     'unfair' under another, on the EXACT SAME data.")
elif abs(dp_difference) > 0.05 and abs(tpr_diff) < 0.05 and abs(fpr_diff) < 0.05:
    print("  Equalized Odds looks satisfied, but Demographic Parity is violated --")
    print("  the reverse pattern from the deck's mini-lab example.")
else:
    print("  Both metrics show some disagreement in this run -- re-run Section 1's")
    print("  data generation with a different random seed to see the pattern shift,")
    print("  since these are randomly generated toy numbers.")



## Section 5 — Why This Happens Mathematically

The reason these metrics can conflict: Group A and Group B have DIFFERENT underlying
qualification rates in our ground truth (40% vs. 20%). Forcing equal APPROVAL rates
(Demographic Parity) when the underlying qualification rates genuinely differ necessarily means
treating similarly-qualified individuals from each group differently in terms of true/false
positive rates -- which is exactly what Equalized Odds measures. This isn't a bug in either
metric -- it's a mathematical consequence of the groups having different base rates, exactly as
the deck states.


In [ ]:

base_rate_a = df[df["group"] == "A"]["ground_truth"].mean()
base_rate_b = df[df["group"] == "B"]["ground_truth"].mean()

print(f"Group A base (qualification) rate: {base_rate_a:.3f}")
print(f"Group B base (qualification) rate: {base_rate_b:.3f}")
print(f"Base rate difference: {abs(base_rate_a - base_rate_b):.3f}")
print("\nWhen base rates differ this much, satisfying Demographic Parity AND Equalized Odds")
print("simultaneously becomes mathematically very difficult -- a genuine, well-documented")
print("result in the fairness literature, now visible in your own generated numbers.")



## Section 6 — Try It Yourself

Change the qualification rates in Section 1 to be EQUAL between groups (e.g. both 30%),
re-run Sections 1-4, and see whether the two metrics agree more closely when base rates match.


In [ ]:

# Example: try setting both groups to the same qualified_rate and re-running
equal_rate_a_truth, equal_rate_a_pred = generate_toy_data(n=100, qualified_rate=0.30)
equal_rate_b_truth, equal_rate_b_pred = generate_toy_data(n=100, qualified_rate=0.30)

equal_df = pd.DataFrame({
    "group": ["A"] * 100 + ["B"] * 100,
    "ground_truth": np.concatenate([equal_rate_a_truth, equal_rate_b_truth]),
    "prediction": np.concatenate([equal_rate_a_pred, equal_rate_b_pred]),
})

equal_dp = demographic_parity(equal_df)
equal_eo = equalized_odds(equal_df)

print("With EQUAL base rates between groups:")
print(f"  Demographic Parity difference: {equal_dp['A'] - equal_dp['B']:+.3f}")
print(f"  Equalized Odds TPR difference: {equal_eo['A']['TPR'] - equal_eo['B']['TPR']:+.3f}")
print(f"  Equalized Odds FPR difference: {equal_eo['A']['FPR'] - equal_eo['B']['FPR']:+.3f}")
print("\nWith matched base rates, the two fairness definitions tend to agree much more closely.")



## Key Takeaways

1. **You just computed, with your own generated numbers**, the deck's central claim: the SAME
   system can look fair by one formal definition and unfair by another, simultaneously and
   correctly by each definition's own math.
2. **This isn't a measurement error or a bug** -- it's a direct mathematical consequence of the
   two groups having different underlying base rates, demonstrated concretely in Section 5.
3. **No single metric is "the" fairness metric** -- Section 4's interpretation step is exactly
   the deck's point that choosing which definition matters requires understanding the actual
   use case and its real-world stakes, not defaulting to whichever number looks best.
4. **Always report which metric was used and why** -- an unqualified claim that a system "is
   fair," without naming the specific metric, is not a complete or honest claim.

**This completes Day 4's hands-on practicals.** Combined with the EU AI Act and GDPR worksheets
(discussion/classification exercises, not code), Day 4's full stack — guardrails, gateway
routing, continuous evaluation, and fairness measurement — is now runnable, closing out all
4 days of hands-on practicals for this training program.
